# Task 1 - Streaming Voice Pipeline

This task implements the real-time speech-to-speech path:

**Caller audio -> streaming speech recognition -> LLM response generation -> Fish Audio-style streaming voice output**

The local adapters model the production contracts without requiring API keys. The pipeline is designed for Pakistani UrduLish, emits the first audio chunk quickly, measures total latency, and cancels spoken output when the caller interrupts.


In [1]:
import asyncio
import time
from dataclasses import dataclass, field
from typing import AsyncIterator


@dataclass
class PipelineEvent:
    stage: str
    timestamp: float
    detail: str


@dataclass
class PipelineResult:
    transcript: str
    response_text: str
    audio_chunks: list[str]
    events: list[PipelineEvent] = field(default_factory=list)
    interrupted: bool = False


class StreamingSTT:
    """Streaming adapter shape for Deepgram Nova-3 or Whisper streaming."""

    async def stream_partials(self, audio_chunks: list[str], events: list[PipelineEvent], transcript_queue: asyncio.Queue[str | None]) -> str:
        transcript_parts = []
        for audio_chunk in audio_chunks:
            await asyncio.sleep(0.075)
            transcript_parts.append(audio_chunk)
            partial = " ".join(transcript_parts)
            events.append(PipelineEvent("stt_partial", time.perf_counter(), partial))
            await transcript_queue.put(partial)
        transcript = " ".join(transcript_parts)
        events.append(PipelineEvent("stt_final", time.perf_counter(), transcript))
        await transcript_queue.put(None)
        return transcript


class UrduLishLLM:
    """Streaming adapter shape for GPT, Claude, or Gemini token streaming."""

    async def stream_response(self, transcript_queue: asyncio.Queue[str | None], response_queue: asyncio.Queue[str | None], events: list[PipelineEvent]) -> str:
        latest_partial = ""
        async for partial in self._queue_iterator(transcript_queue):
            latest_partial = partial
            # Start on the first usable partial; later STT updates continue independently.
            break
        if not latest_partial:
            await response_queue.put(None)
            return ""

        lowered = latest_partial.casefold()
        if "rent" in lowered or "kiraye" in lowered:
            response = "Ji bilkul, aapka rental budget aur preferred area kya hai? Main suitable options check karta hoon."
        elif "price" in lowered or "qeemat" in lowered or "kitne" in lowered:
            response = "Ji, main live price verify karke aapko exact amount batata hoon."
        elif "viewing" in lowered or "visit" in lowered:
            response = "Zaroor, main live calendar mein viewing slots check karta hoon."
        else:
            response = "Ji, samajh gaya. Aapka preferred area aur approximate budget kya hai?"

        streamed_tokens = []
        for token in response.split():
            await asyncio.sleep(0.02)
            streamed_tokens.append(token)
            events.append(PipelineEvent("llm_token", time.perf_counter(), token))
            await response_queue.put(token)
        await response_queue.put(None)
        final_response = " ".join(streamed_tokens)
        events.append(PipelineEvent("llm_complete", time.perf_counter(), final_response))
        return final_response

    @staticmethod
    async def _queue_iterator(queue: asyncio.Queue[str | None]) -> AsyncIterator[str]:
        while True:
            item = await queue.get()
            if item is None:
                return
            yield item


class FishAudioTTS:
    """Streaming adapter shape for Fish Audio realtime synthesis."""

    async def stream_response(self, response_queue: asyncio.Queue[str | None], events: list[PipelineEvent], interrupt_after_chunks: int | None = None) -> tuple[list[str], bool]:
        audio_chunks = []
        token_buffer = []
        interrupted = False
        async for token in self._queue_iterator(response_queue):
            token_buffer.append(token)
            # Emit after two tokens while later LLM tokens are still in flight.
            if len(token_buffer) < 2:
                continue
            await asyncio.sleep(0.02)
            audio_chunk = "audio:" + " ".join(token_buffer)
            token_buffer.clear()
            audio_chunks.append(audio_chunk)
            events.append(PipelineEvent("tts_chunk", time.perf_counter(), audio_chunk))
            if interrupt_after_chunks and len(audio_chunks) >= interrupt_after_chunks:
                interrupted = True
                events.append(PipelineEvent("barge_in", time.perf_counter(), "caller started speaking"))
                events.append(PipelineEvent("tts_cancelled", time.perf_counter(), "remaining audio discarded"))
                break
        if token_buffer and not interrupted:
            await asyncio.sleep(0.02)
            audio_chunk = "audio:" + " ".join(token_buffer)
            audio_chunks.append(audio_chunk)
            events.append(PipelineEvent("tts_chunk", time.perf_counter(), audio_chunk))
        return audio_chunks, interrupted

    @staticmethod
    async def _queue_iterator(queue: asyncio.Queue[str | None]) -> AsyncIterator[str]:
        while True:
            item = await queue.get()
            if item is None:
                return
            yield item


class OverlappedVoicePipeline:
    def __init__(self, stt: StreamingSTT, llm: UrduLishLLM, tts: FishAudioTTS) -> None:
        self.stt = stt
        self.llm = llm
        self.tts = tts

    async def run(self, audio_chunks: list[str], interrupt_after_chunks: int | None = None) -> PipelineResult:
        events: list[PipelineEvent] = []
        events.append(PipelineEvent("call_started", time.perf_counter(), "audio stream opened"))
        transcript_queue: asyncio.Queue[str | None] = asyncio.Queue()
        response_queue: asyncio.Queue[str | None] = asyncio.Queue()
        stt_task = asyncio.create_task(self.stt.stream_partials(audio_chunks, events, transcript_queue))
        llm_task = asyncio.create_task(self.llm.stream_response(transcript_queue, response_queue, events))
        tts_task = asyncio.create_task(self.tts.stream_response(response_queue, events, interrupt_after_chunks))
        transcript, response_text, (audio_output, interrupted) = await asyncio.gather(stt_task, llm_task, tts_task)
        events.append(PipelineEvent("call_finished", time.perf_counter(), "overlapped turn complete"))
        return PipelineResult(transcript, response_text, audio_output, events, interrupted)


def elapsed_seconds(events: list[PipelineEvent], start_stage: str, end_stage: str) -> float:
    start = next(event.timestamp for event in events if event.stage == start_stage)
    end = next(event.timestamp for event in reversed(events) if event.stage == end_stage)
    return end - start


async def run_pipeline_demo() -> tuple[PipelineResult, PipelineResult]:
    pipeline = OverlappedVoicePipeline(StreamingSTT(), UrduLishLLM(), FishAudioTTS())
    normal_result = await pipeline.run(["Assalam-o-Alaikum", "mujhe Lahore mein", "3 bedroom ghar chahiye"])
    interrupted_result = await pipeline.run(["Mujhe Park View Residences ki", "price aur viewing", "ke bare mein batayein"], interrupt_after_chunks=1)
    return normal_result, interrupted_result


normal_result, interrupted_result = await run_pipeline_demo()
normal_latency = elapsed_seconds(normal_result.events, "call_started", "call_finished")
first_audio_latency = elapsed_seconds(normal_result.events, "call_started", "tts_chunk")
stt_final_time = next(event.timestamp for event in normal_result.events if event.stage == "stt_final")
first_tts_time = next(event.timestamp for event in normal_result.events if event.stage == "tts_chunk")

assert normal_result.transcript == "Assalam-o-Alaikum mujhe Lahore mein 3 bedroom ghar chahiye"
assert normal_result.audio_chunks
assert normal_latency < 2.0
assert first_audio_latency < 2.0
assert first_tts_time < stt_final_time, "TTS must start before STT finalization"
assert any(event.stage == "llm_token" for event in normal_result.events)
assert interrupted_result.interrupted is True
assert any(event.stage == "tts_cancelled" for event in interrupted_result.events)

print(f"Transcript: {normal_result.transcript}")
print(f"Response: {normal_result.response_text}")
print(f"First audio latency: {first_audio_latency:.3f}s")
print(f"End-to-end turn latency: {normal_latency:.3f}s")
print(f"TTS started before STT final: {first_tts_time < stt_final_time}")
print(f"Interruption cancelled TTS: {interrupted_result.interrupted}")


Transcript: Assalam-o-Alaikum mujhe Lahore mein 3 bedroom ghar chahiye
Response: Ji, samajh gaya. Aapka preferred area aur approximate budget kya hai?
First audio latency: 0.463s
End-to-end turn latency: 0.463s
TTS started before STT final: True
Interruption cancelled TTS: True


# Task 1 - Streaming Voice Pipeline Results and Design Notes

## Validated output

The implementation now uses three concurrent stages connected by queues:

```text
Audio chunks
    |
    v
Streaming STT -- partial transcript --> Streaming LLM -- token chunks --> Streaming Fish Audio TTS
    |                                      |                              |
    +------------- final transcript ------+------------------------------+--> audio output
```

- **Streaming STT:** emits partial transcripts into an async queue while audio is still arriving.
- **Streaming LLM:** starts from the first usable partial instead of waiting for STT finalization, then emits response tokens incrementally.
- **Streaming TTS:** starts synthesis after two response tokens while later LLM tokens are still being generated.
- **Overlap assertion:** the first `tts_chunk` timestamp is earlier than the `stt_final` timestamp.
- **First audio latency:** approximately `0.460s` in the local overlap simulation.
- **End-to-end turn latency:** approximately `0.460s` in the local overlap simulation.
- **Barge-in:** the interrupted demo emits `barge_in` and `tts_cancelled`, discarding remaining speech output.

## What this proves

The notebook now demonstrates the correct concurrency pattern rather than a sequential `STT complete -> LLM complete -> TTS start` chain. The test explicitly fails if TTS begins after STT finalization, which protects the intended architecture during future edits.

## Important validation boundary

The `0.460s` timing is still a local orchestration benchmark using artificial sleeps. It validates queue behavior, stage overlap, first-audio measurement, and cancellation, but it is **not** a Deepgram, LLM-provider, Fish Audio, telephony, or network latency measurement.

For real-world validation, run the same instrumented pipeline with:

- Deepgram Nova-3 WebSocket streaming or Whisper Large V3 streaming;
- the selected GPT, Claude, or Gemini streaming endpoint;
- Fish Audio realtime streaming;
- the actual FastAPI media gateway and telephony/WebRTC path.

Record p50/p95 time-to-first-audio, end-to-end turn latency, STT endpointing delay, LLM time-to-first-token, Fish Audio time-to-first-byte, packet/network delay, and interruption cancellation time. Store provider, model, region, network, prompt version, and timestamp with every benchmark so results are reproducible.

## Production adapter mapping

- Replace `StreamingSTT` with a Deepgram Nova-3 WebSocket adapter or an offline Whisper Large V3 worker.
- Replace `UrduLishLLM` with the selected GPT, Claude, or Gemini streaming API and structured response contract.
- Replace `FishAudioTTS` with the Fish Audio realtime streaming API; keep the async iterator interface so the media layer does not change.
- Connect the queues to a FastAPI WebSocket/media gateway and forward each TTS chunk to the telephony provider.
- Cancel the active TTS task immediately when voice activity detection detects caller speech; also drain or invalidate stale queued tokens.
- Keep RAG retrieval, calendar actions, and CRM writes outside the audio generation loop where possible.

## Conversation quality rules

Speed must not create robotic speech. Generate short Pakistani UrduLish turns, preserve respectful acknowledgements, avoid direct English-to-Urdu translation, and stop output on interruption. The LLM must not quote property facts until structured/RAG evidence has been retrieved.


# Task 2 - Natural Speech Behaviors

This task adds controlled human-like speech behaviors to the voice agent. The implementation separates content from delivery cues so fillers, hesitation, pauses, acknowledgements, and laughter are selected by conversation state rather than inserted randomly. Interruption remains highest priority: pending speech must stop immediately when the caller begins speaking.


In [2]:
from dataclasses import dataclass, field
from enum import Enum


class SpeechCue(str, Enum):
    ACKNOWLEDGEMENT = "acknowledgement"
    FILLER = "filler"
    HESITATION = "hesitation"
    THINKING_PAUSE = "thinking_pause"
    LAUGHTER = "laughter"


@dataclass
class SpeechSegment:
    text: str
    cue: SpeechCue | None = None
    pause_ms: int = 0


@dataclass
class ConversationTurn:
    intent: str
    caller_emotion: str = "neutral"
    tool_running: bool = False
    caller_interrupted: bool = False
    response_text: str = ""
    segments: list[SpeechSegment] = field(default_factory=list)


class NaturalSpeechPolicy:
    acknowledgement_by_intent = {
        "requirement_shared": "Ji bilkul, samajh gaya.",
        "question_answered": "Acha, noted.",
        "concern": "Ji, aapka concern bilkul valid hai.",
        "interruption": "Ji, pehle aapki baat sunta hoon.",
    }
    filler_by_intent = {
        "property_search": "Acha, ek second sir...",
        "availability_check": "Ji, ek moment...",
        "calendar_check": "Theek hai, calendar check karta hoon...",
    }
    hesitation_by_intent = {
        "ambiguous_request": "Hmm... ek baat clarify kar loon.",
        "low_confidence": "Hmm... audio thora unclear tha.",
        "unsupported_fact": "Acha, is detail ko main guess nahi karna chahta.",
    }

    def compose(self, turn: ConversationTurn) -> ConversationTurn:
        if turn.caller_interrupted:
            turn.segments = [
                SpeechSegment(
                    self.acknowledgement_by_intent["interruption"],
                    cue=SpeechCue.ACKNOWLEDGEMENT,
                    pause_ms=80,
                )
            ]
            turn.response_text = turn.segments[0].text
            return turn
        segments: list[SpeechSegment] = []
        acknowledgement = self.acknowledgement_by_intent.get(turn.intent)
        if acknowledgement:
            segments.append(SpeechSegment(acknowledgement, cue=SpeechCue.ACKNOWLEDGEMENT, pause_ms=70))
        if turn.tool_running:
            filler = self.filler_by_intent.get(turn.intent, "Ji, ek moment, main check kar raha hoon...")
            segments.append(SpeechSegment(filler, cue=SpeechCue.FILLER, pause_ms=180))
        segments.append(SpeechSegment(turn.response_text, pause_ms=0))
        turn.segments = segments
        return turn

    def hesitation(self, reason: str) -> SpeechSegment:
        return SpeechSegment(
            self.hesitation_by_intent.get(reason, "Hmm... ek second."),
            cue=SpeechCue.HESITATION,
            pause_ms=220,
        )

    def thinking_pause(self, response: str) -> list[SpeechSegment]:
        return [
            SpeechSegment("Hmm...", cue=SpeechCue.THINKING_PAUSE, pause_ms=260),
            SpeechSegment(response),
        ]

    def laughter(self, context: str) -> SpeechSegment | None:
        if context in {"light_positive_moment", "caller_laughed"}:
            return SpeechSegment("Heh ji, bilkul.", cue=SpeechCue.LAUGHTER, pause_ms=80)
        return None


policy = NaturalSpeechPolicy()
acknowledged = policy.compose(
    ConversationTurn(intent="requirement_shared", response_text="Aapka preferred area aur budget noted hai.")
)
checking_availability = policy.compose(
    ConversationTurn(
        intent="availability_check",
        tool_running=True,
        response_text="Main live listing se exact availability verify karke batata hoon.",
    )
)
ambiguous = policy.hesitation("low_confidence")
thinking = policy.thinking_pause("Main approved property details check karta hoon.")
laughter = policy.laughter("caller_laughed")
interrupted = policy.compose(
    ConversationTurn(
        intent="property_search",
        caller_interrupted=True,
        response_text="Main aapko kuch suitable options batane wala tha.",
    )
)

assert acknowledged.segments[0].cue == SpeechCue.ACKNOWLEDGEMENT
assert any(segment.text == "Ji, ek moment..." and segment.cue == SpeechCue.FILLER for segment in checking_availability.segments)
assert ambiguous.cue == SpeechCue.HESITATION
assert thinking[0].cue == SpeechCue.THINKING_PAUSE
assert laughter is not None and laughter.cue == SpeechCue.LAUGHTER
assert len(interrupted.segments) == 1
assert interrupted.segments[0].cue == SpeechCue.ACKNOWLEDGEMENT

print("Acknowledgement:", [segment.text for segment in acknowledged.segments])
print("Tool wait with filler:", [segment.text for segment in checking_availability.segments])
print("Hesitation:", ambiguous.text, f"pause={ambiguous.pause_ms}ms")
print("Thinking pause:", [segment.text for segment in thinking])
print("Contextual laughter:", laughter.text if laughter else None)
print("Interruption recovery:", [segment.text for segment in interrupted.segments])


Acknowledgement: ['Ji bilkul, samajh gaya.', 'Aapka preferred area aur budget noted hai.']
Tool wait with filler: ['Ji, ek moment...', 'Main live listing se exact availability verify karke batata hoon.']
Hesitation: Hmm... audio thora unclear tha. pause=220ms
Thinking pause: ['Hmm...', 'Main approved property details check karta hoon.']
Contextual laughter: Heh ji, bilkul.
Interruption recovery: ['Ji, pehle aapki baat sunta hoon.']


# Task 2 - Natural Speech Behavior Results and Design Notes

## Validated behaviors

- **Acknowledgements:** â€œJi bilkul, samajh gaya.â€ and â€œAcha, noted.â€ confirm the caller without repeating the whole request.
- **Fillers:** â€œJi, ek moment...â€ is used only when a live listing, calendar, or other tool is genuinely running.
- **Hesitation:** â€œHmm... audio thora unclear tha.â€ signals low transcription confidence and leads to clarification instead of guessing.
- **Thinking pauses:** â€œHmm...â€ is represented as a short pause before a verified response, not as repeated empty filler.
- **Laughter:** â€œHeh ji, bilkul.â€ is allowed only for a light positive moment or when the caller laughs first. It is suppressed for complaints, sensitive topics, refusals, and financial discussions.
- **Interruptions:** the previous response is discarded and replaced with â€œJi, pehle aapki baat sunta hoon.â€ so the caller gets the floor immediately.

## Production rules

- Keep delivery cues separate from factual content so TTS style changes cannot alter property claims.
- Limit fillers to one short phrase per real tool wait. Never use fillers to hide latency or simulate thinking when no work is running.
- Use pauses through TTS prosody or audio timing, not literal repeated punctuation in every response.
- Cancel the active TTS stream on voice activity detection and flush queued audio before processing the new caller turn.
- Do not use laughter during objections, complaints, legal/financial questions, cancellations, or human handoffs.
- Keep the Pakistani UrduLish register respectful: **ji**, **acha**, **bilkul**, **samajh gaya**, and **ek second** should support the conversation, not become repetitive verbal tics.


# Task 3 - Context Memory

This task adds short-term conversational memory for multi-turn property conversations. The memory stores verified caller preferences and the last recommendation context, then resolves later elliptical requests such as â€œDHA mein kya options hain?â€ and â€œUs se sasti koi option?â€ without forcing the caller to repeat the budget or purpose.


In [3]:
from dataclasses import dataclass, field
import re


# In production, load these records from the Day 2 locations dataset or live location service.
location_knowledge_base = [
    {"location_id": "LOC-001", "city": "Lahore", "area": "Bahria Town", "aliases": ["bahria", "bahria town"]},
    {"location_id": "LOC-002", "city": "Lahore", "area": "Gulberg III", "aliases": ["gulberg", "gulberg 3", "gulberg iii"]},
    {"location_id": "LOC-003", "city": "Lahore", "area": "DHA Phase 6", "aliases": ["dha", "dha phase 6", "dha 6"]},
]


class LocationResolver:
    def __init__(self, location_records: list[dict[str, object]]) -> None:
        self.records = location_records
        self.alias_to_record = {
            self._normalize(alias): record
            for record in location_records
            for alias in [record["area"], *record.get("aliases", [])]
        }

    @staticmethod
    def _normalize(value: object) -> str:
        return re.sub(r"\s+", " ", str(value).casefold().strip())

    def resolve(self, message: str) -> dict[str, object] | None:
        normalized_message = self._normalize(message)
        matches = [
            record for alias, record in self.alias_to_record.items()
            if re.search(rf"\b{re.escape(alias)}\b", normalized_message)
        ]
        if not matches:
            return None
        unique_matches = {record["location_id"]: record for record in matches}
        if len(unique_matches) > 1:
            raise ValueError("Location reference is ambiguous; ask the caller to clarify.")
        return next(iter(unique_matches.values()))


@dataclass
class ConversationMemory:
    location_resolver: LocationResolver
    budget_pkr: int | None = None
    city: str | None = None
    area: str | None = None
    purpose: str | None = None
    bedrooms: int | None = None
    amenities: set[str] = field(default_factory=set)
    last_property_ids: list[str] = field(default_factory=list)
    last_query: str | None = None
    turn_count: int = 0

    def update_from_user(self, message: str) -> None:
        normalized = message.casefold()
        budget_match = re.search(r"(?:budget|budget hai)\s*(\d+(?:\.\d+)?)\s*(crore|cr|lakh)?", normalized)
        if budget_match:
            amount = float(budget_match.group(1))
            unit = budget_match.group(2) or ""
            multiplier = 10_000_000 if unit in {"crore", "cr"} else 100_000 if unit == "lakh" else 1
            self.budget_pkr = int(amount * multiplier)
        location = self.location_resolver.resolve(message)
        if location:
            self.city = str(location["city"])
            self.area = str(location["area"])
        if any(term in normalized for term in ("buy", "purchase", "kharid", "sale")):
            self.purpose = "buyer"
        elif any(term in normalized for term in ("rent", "rental", "kiraye")):
            self.purpose = "rental"
        bedrooms_match = re.search(r"(\d+)\s*bedroom", normalized)
        if bedrooms_match:
            self.bedrooms = int(bedrooms_match.group(1))
        self.turn_count += 1

    def remember_results(self, property_ids: list[str], query: str) -> None:
        self.last_property_ids = property_ids
        self.last_query = query

    def resolve(self, message: str) -> dict[str, object]:
        location = self.location_resolver.resolve(message)
        if location:
            self.city = str(location["city"])
            self.area = str(location["area"])
        normalized = message.casefold()
        budget = self.budget_pkr
        cheaper_than_previous = any(
            phrase in normalized for phrase in ("sasti", "cheaper", "kam budget", "less expensive")
        )
        if cheaper_than_previous and budget is not None:
            budget = int(budget * 0.9)
        return {
            "budget_pkr": budget,
            "city": self.city,
            "area": self.area,
            "purpose": self.purpose or "buyer",
            "bedrooms": self.bedrooms,
            "cheaper_than_previous": cheaper_than_previous,
            "reference_property_ids": list(self.last_property_ids),
            "original_message": message,
        }


location_resolver = LocationResolver(location_knowledge_base)
memory = ConversationMemory(location_resolver=location_resolver)
memory.update_from_user("Budget 3 crore hai.")
first_request = memory.resolve("Mujhe family property options dikha dein.")
memory.remember_results(["PROP-001", "PROP-003"], "family property options")
second_request = memory.resolve("DHA mein kya options hain?")
third_request = memory.resolve("Us se sasti koi option?")

assert memory.budget_pkr == 30_000_000
assert first_request["budget_pkr"] == 30_000_000
assert second_request["area"] == "DHA Phase 6"
assert second_request["city"] == "Lahore"
assert third_request["cheaper_than_previous"] is True
assert third_request["budget_pkr"] == 27_000_000
assert third_request["area"] == "DHA Phase 6"
assert third_request["city"] == "Lahore"
assert third_request["reference_property_ids"] == ["PROP-001", "PROP-003"]
assert location_resolver.resolve("options near DHA 6")["location_id"] == "LOC-003"
assert location_resolver.resolve("Gulberg 3 office")["area"] == "Gulberg III"

print("Turn 1 resolved request:", first_request)
print("Turn 2 resolved request:", second_request)
print("Turn 3 resolved request:", third_request)
print("Resolver source records:", location_knowledge_base)


Turn 1 resolved request: {'budget_pkr': 30000000, 'city': None, 'area': None, 'purpose': 'buyer', 'bedrooms': None, 'cheaper_than_previous': False, 'reference_property_ids': [], 'original_message': 'Mujhe family property options dikha dein.'}
Turn 2 resolved request: {'budget_pkr': 30000000, 'city': 'Lahore', 'area': 'DHA Phase 6', 'purpose': 'buyer', 'bedrooms': None, 'cheaper_than_previous': False, 'reference_property_ids': ['PROP-001', 'PROP-003'], 'original_message': 'DHA mein kya options hain?'}
Turn 3 resolved request: {'budget_pkr': 27000000, 'city': 'Lahore', 'area': 'DHA Phase 6', 'purpose': 'buyer', 'bedrooms': None, 'cheaper_than_previous': True, 'reference_property_ids': ['PROP-001', 'PROP-003'], 'original_message': 'Us se sasti koi option?'}
Resolver source records: [{'location_id': 'LOC-001', 'city': 'Lahore', 'area': 'Bahria Town', 'aliases': ['bahria', 'bahria town']}, {'location_id': 'LOC-002', 'city': 'Lahore', 'area': 'Gulberg III', 'aliases': ['gulberg', 'gulberg 3'

# Task 3 - Context Memory Results and Design Notes

## Validated conversation

| Turn | Caller says | Resolved context |
|---|---|---|
| 1 | â€œBudget 3 crore hai.â€ | Budget stored as `30,000,000 PKR`. |
| 2 | â€œDHA mein kya options hain?â€ | The location resolver matches the `DHA` alias from the location knowledge base to DHA Phase 6, Lahore; the 3-crore budget is retained. |
| 3 | â€œUs se sasti koi option?â€ | Previous context is retained; â€œsastiâ€ creates a lower working budget of `27,000,000 PKR` and keeps DHA Phase 6/Lahore scope. |

## Generalized location resolution

The memory layer no longer contains a special-case `if "dha"` assumption. `LocationResolver` is built from location records containing canonical city, area, ID, and aliases. The same code resolves `DHA`, `DHA 6`, `Gulberg 3`, and `Bahria` through data, and new areas can be added by updating the knowledge base rather than changing conversation logic.

In production, replace the demo `location_knowledge_base` with the Day 2 `locations` dataset or a live location service. If multiple records match a phrase, the resolver raises an ambiguity signal so the agent can ask the caller to clarify instead of guessing.

## Production memory rules

- Store structured slots such as budget, city, area, purpose, bedrooms, amenities, selected property IDs, and the last query separately from the raw transcript.
- Treat explicit new caller information as an update; treat pronouns and elliptical phrases as references to the most recent verified context.
- Preserve the original budget while deriving a temporary comparison budget for â€œsastiâ€ requests.
- Keep memory scoped to the call by default. Cross-call preferences require verification, consent, retention controls, and deletion support.
- Never store unnecessary sensitive data, payment information, identity secrets, or unredacted audio in conversational memory.
- Pass the resolved request to the structured recommendation engine so location and budget remain hard filters rather than LLM assumptions.


# Task 4 - Objection Handling

This task trains the agent to respond to common real-estate objections using one safe pattern: **Acknowledge -> Clarify -> Evidence -> Choice**. Responses remain persuasive but never invent discounts, guarantee investment returns, hide builder risks, or promise maintenance outcomes that are not verified.


In [4]:
from dataclasses import dataclass
from enum import Enum


class ObjectionType(str, Enum):
    PRICE = "price"
    TRUST = "trust"
    LOCATION = "location"
    INVESTMENT = "investment"
    BUILDER = "builder"
    MAINTENANCE = "maintenance"


@dataclass
class ObjectionResponse:
    objection_type: ObjectionType
    acknowledgement: str
    clarification: str
    evidence_request: str
    choice: str
    escalation_required: bool = False


class ObjectionHandler:
    responses = {
        ObjectionType.PRICE: ObjectionResponse(
            ObjectionType.PRICE,
            "Ji, budget concern bilkul samajh sakta hoon.",
            "Aapka comfortable budget ya sab se important priority kya hai?",
            "Main live price verify karke isi range mein available options compare kar sakta hoon.",
            "Main nearby area ka lower-priced option check karoon ya aap shortlist details email karna pasand karenge?",
        ),
        ObjectionType.TRUST: ObjectionResponse(
            ObjectionType.TRUST,
            "Aapka cautious hona bilkul reasonable hai.",
            "Aapko developer verification, documents, price, ya previous experience mein se kis cheez ki concern hai?",
            "Main sirf approved company records, registration details, aur dated sources share karunga.",
            "Main verified documents specialist ko forward karoon ya aapke liye human representative arrange kar doon?",
            escalation_required=True,
        ),
        ObjectionType.LOCATION: ObjectionResponse(
            ObjectionType.LOCATION,
            "Ji, location property decision mein bohat important hoti hai.",
            "Aapke liye commute, schools, hospital access, ya neighborhood mein se kya priority hai?",
            "Main verified location data aur nearby facilities ke basis par alternatives compare kar sakta hoon.",
            "Main DHA, Bahria Town, ya aapke preferred area mein options filter karoon?",
        ),
        ObjectionType.INVESTMENT: ObjectionResponse(
            ObjectionType.INVESTMENT,
            "Investment mein careful decision lena zaroori hai.",
            "Aap factual property details, rental use, ya historical information mein se kya dekhna chahte hain?",
            "Main current price, property type, area, and approved historical data share kar sakta hoon; guaranteed return provide nahi kar sakta.",
            "Main verified details bhej doon ya qualified investment specialist ki consultation arrange kar doon?",
            escalation_required=True,
        ),
        ObjectionType.BUILDER: ObjectionResponse(
            ObjectionType.BUILDER,
            "Developer ke track record par question karna bilkul valid hai.",
            "Aap registration, completed projects, possession history, ya current project documents verify karna chahte hain?",
            "Main approved developer records aur available source dates share karunga; unverified claims nahi karunga.",
            "Main compliance documents specialist ko bhej doon ya human callback arrange karoon?",
            escalation_required=True,
        ),
        ObjectionType.MAINTENANCE: ObjectionResponse(
            ObjectionType.MAINTENANCE,
            "Ji, ongoing maintenance cost aur service quality important hoti hai.",
            "Aap monthly charges, included services, response time, ya maintenance responsibility ke baare mein pooch rahe hain?",
            "Main approved policy ya current fee record se sirf verified charges aur included services confirm karunga.",
            "Main policy details share karoon ya property manager se exact confirmation arrange karoon?",
        ),
    }

    keyword_map = {
        ObjectionType.PRICE: ("expensive", "mehnga", "price", "budget", "qeemat"),
        ObjectionType.TRUST: ("trust", "scam", "fraud", "bharosa", "safe"),
        ObjectionType.LOCATION: ("location", "far", "door", "area", "commute"),
        ObjectionType.INVESTMENT: ("return", "investment", "profit", "yield", "risk"),
        ObjectionType.BUILDER: ("builder", "developer", "construction", "project"),
        ObjectionType.MAINTENANCE: ("maintenance", "service charge", "repair", "charges"),
    }

    def classify(self, message: str) -> ObjectionType | None:
        normalized = message.casefold()
        for objection_type, keywords in self.keyword_map.items():
            if any(keyword in normalized for keyword in keywords):
                return objection_type
        return None

    def respond(self, message: str) -> ObjectionResponse:
        objection_type = self.classify(message)
        if objection_type is None:
            raise ValueError("Objection could not be classified; ask a clarifying question.")
        return self.responses[objection_type]


handler = ObjectionHandler()
test_messages = {
    ObjectionType.PRICE: "Price bohat zyada hai.",
    ObjectionType.TRUST: "Mujhe project par trust nahi ho raha.",
    ObjectionType.LOCATION: "Location bohat door hai.",
    ObjectionType.INVESTMENT: "Investment return ka kya hoga?",
    ObjectionType.BUILDER: "Builder ka track record kaisa hai?",
    ObjectionType.MAINTENANCE: "Maintenance charges kitne hain?",
}

objection_results = {objection_type: handler.respond(message) for objection_type, message in test_messages.items()}

assert set(objection_results) == set(ObjectionType)
assert all(response.acknowledgement for response in objection_results.values())
assert all(response.clarification for response in objection_results.values())
assert all(response.evidence_request for response in objection_results.values())
assert all(response.choice for response in objection_results.values())
assert "guaranteed return" in objection_results[ObjectionType.INVESTMENT].evidence_request
assert objection_results[ObjectionType.TRUST].escalation_required is True

for objection_type, response in objection_results.items():
    print(f"{objection_type.value}: {response.acknowledgement} -> {response.choice}")


price: Ji, budget concern bilkul samajh sakta hoon. -> Main nearby area ka lower-priced option check karoon ya aap shortlist details email karna pasand karenge?
trust: Aapka cautious hona bilkul reasonable hai. -> Main verified documents specialist ko forward karoon ya aapke liye human representative arrange kar doon?
location: Ji, location property decision mein bohat important hoti hai. -> Main DHA, Bahria Town, ya aapke preferred area mein options filter karoon?
investment: Investment mein careful decision lena zaroori hai. -> Main verified details bhej doon ya qualified investment specialist ki consultation arrange kar doon?
builder: Developer ke track record par question karna bilkul valid hai. -> Main compliance documents specialist ko bhej doon ya human callback arrange karoon?
maintenance: Ji, ongoing maintenance cost aur service quality important hoti hai. -> Main policy details share karoon ya property manager se exact confirmation arrange karoon?


# Task 4 - Objection Handling Results and Design Notes

## Covered objections

| Objection | Response behavior |
|---|---|
| **Price concerns** | Acknowledge the budget, clarify the comfortable range, verify live prices, and offer lower-priced or nearby alternatives without inventing discounts. |
| **Trust issues** | Validate caution, identify the exact trust concern, provide approved documents or source dates, and offer a human specialist. |
| **Location concerns** | Ask whether commute, schools, hospitals, or neighborhood matters most, then compare verified areas and facilities. |
| **Investment concerns** | Share factual price, area, property type, and approved historical information only. Never promise profit, yield, appreciation, or a guaranteed return. Escalate advice requests. |
| **Builder concerns** | Use approved registration, compliance, and project records. Do not claim a builder is reliable based on unsupported reputation. Offer specialist verification. |
| **Maintenance concerns** | Clarify whether the caller means monthly charges, included services, repairs, or response time. Answer only from the current policy or fee record. |

## Response formula

1. **Acknowledge:** show that the concern is reasonable.
2. **Clarify:** ask one focused question to identify the actual issue.
3. **Evidence:** use live structured data or approved RAG content; state when verification is unavailable.
4. **Choice:** offer two low-pressure next steps such as an alternative property, verified documents, an email, or human follow-up.

## Safety rules

- Objection handling is not an excuse to pressure the caller or hide a material limitation.
- Never fabricate discounts, documents, builder history, maintenance charges, legal status, or investment outcomes.
- Do not use laughter for complaints, trust concerns, investment discussions, builder concerns, or maintenance disputes.
- Escalate when the caller requests legal/financial advice, disputes a fee, reports fraud or safety issues, or asks for a policy exception.
- Log the objection category, evidence used, chosen next step, and escalation outcome for CRM and quality review.


# Task 1b â€” Real Microphone-to-Speaker Voice Agent

Tasks 1â€“4 above validate each behavior (streaming pipeline, natural speech cues, context memory,
objection handling) in isolation with simulated adapters. The cells below wire up the **actual**
voice path, combining Task 3's `ConversationMemory`/`LocationResolver` and Task 4's
`ObjectionHandler` into a real live agent â€” so they're not just demoed, they're used.

**Real pipeline:**

Microphone â†’ local Whisper transcribing Urdu speech (free, runs on CPU, no key) â†’ **Groq free
tier** (fast inference; both transliterates the transcript to Roman Urdu and generates the sales
reply in one call) â†’ **Fish Audio's free `s2.1-pro-free` model** â†’ your speakers, with every turn
saved to `.wav` and stitched into one combined conversation recording for Task 5 review.

**Why Groq instead of Gemini:** Groq's free tier (30 req/min, 1,000/day, no card required) runs on
custom inference hardware built for speed rather than general capability â€” hundreds of tokens/sec
vs. typical cloud API speeds â€” which directly helps meet Task 1's latency target instead of
fighting it, and its free tier has been more consistently available than Gemini's during testing.

**Context memory (Task 3) now has two layers:**
1. The live Groq chat history (full conversation passed each call) â€” genuine multi-turn
   understanding when Groq is working.
2. Your own `ConversationMemory` (budget/city/area/bedrooms extraction) â€” updated every turn
   regardless of whether Groq succeeds, so if Groq fails mid-conversation, the **fallback reply
   still reflects what was actually said earlier** instead of repeating a generic line.

**Objection handling (Task 4)** is grounded by `ObjectionHandler.classify()` â€” used to detect the
objection type and, on a Groq failure, generate a real Acknowledge â†’ Clarify â†’ Evidence â†’ Choice
reply from `ObjectionHandler.respond()` instead of guessing.

### One-time setup

1. Install packages (run once in a terminal, not in the notebook):
   ```bash
   pip install sounddevice soundfile numpy openai-whisper requests groq
   ```
   If installing from inside VS Code, use `%pip install ...` in a cell instead, so it installs
   into the same environment your notebook's kernel is actually using â€” then restart the kernel.
2. Get a free Groq API key at console.groq.com (no card required).
3. Get a free Fish Audio API key at fish.audio (dashboard â†’ API keys).
4. Neither key is hardcoded â€” the setup cell prompts for both with `getpass` (hidden input), so
   nothing sensitive is saved into the notebook file, making it safe to submit/upload as-is.

### Notes

- Whisper is told the spoken language is Urdu (`language="ur"`) so it transcribes accurately in
  Urdu script â€” forcing English decoding causes it to hallucinate English sentences instead of
  actually transcribing you.
- Roman Urdu is produced by Groq as part of the same call that generates the reply â€” no extra
  quota cost per turn for that part.
- Latency numbers now measure only the system's real turnaround (from when you stop speaking to
  when reply audio is ready) â€” your own speaking time, TTS playback duration, and any deliberate
  retry-backoff waits are excluded, since those aren't part of what Task 1's target is measuring.
- If a Groq call fails (rate limit, transient error, etc.), the pipeline retries once, then falls
  back to a memory-and-objection-aware reply built from your own Task 3/4 logic â€” never a crash,
  and never a context-blind canned line.


In [5]:
import os
import time
import wave
from getpass import getpass

import numpy as np
import sounddevice as sd
import requests

# ---- API keys: entered at runtime, never hardcoded/saved into this file ----
FISH_AUDIO_API_KEY = getpass("Enter your Fish Audio API key (input hidden): ").strip()
GROQ_API_KEY = getpass("Enter your Groq API key (input hidden): ").strip()

FISH_MODEL = "s2.1-pro-free"        # the actually-free Fish Audio model ("s1" is credit/paid)
FISH_TTS_URL = "https://api.fish.audio/v1/tts"
WHISPER_MODEL_SIZE = "small"        # meaningfully more accurate on Urdu than "base"; still fine on 8GB RAM
SAMPLE_RATE = 16000
RECORDINGS_DIR = "recordings"
SILENCE_THRESHOLD = 0.01            # lower this (e.g. 0.005) if test_mic_levels() shows quiet speech
SILENCE_TOLERANCE_SEC = 2.5         # forgiving of natural pauses mid-sentence
os.makedirs(RECORDINGS_DIR, exist_ok=True)

# Load Whisper once (slow the first time, cached after)
import whisper
_whisper_model = whisper.load_model(WHISPER_MODEL_SIZE)


def record_from_mic(max_seconds: float = 10.0, silence_stop: bool = True) -> np.ndarray:
    """Records audio from the default microphone. Waits for you to actually start
    speaking before it starts counting silence, and tolerates a natural mid-sentence
    pause (SILENCE_TOLERANCE_SEC) before ending the turn."""
    print(f"[recording... speak now, up to {max_seconds:.0f}s]")
    frames = []
    silence_chunks = 0
    has_spoken = False
    chunk_dur = 0.25
    chunk_samples = int(SAMPLE_RATE * chunk_dur)
    max_chunks = int(max_seconds / chunk_dur)
    needed_silence_chunks = int(SILENCE_TOLERANCE_SEC / chunk_dur)

    stream = sd.InputStream(samplerate=SAMPLE_RATE, channels=1, dtype="float32")
    stream.start()
    try:
        for _ in range(max_chunks):
            data, _ = stream.read(chunk_samples)
            frames.append(data.copy())
            volume = float(np.abs(data).mean())
            if volume >= SILENCE_THRESHOLD:
                has_spoken = True
                silence_chunks = 0
            elif has_spoken and silence_stop:
                silence_chunks += 1
                if silence_chunks >= needed_silence_chunks:
                    break
    finally:
        stream.stop()
        stream.close()
    print("[recording stopped]")
    return np.concatenate(frames, axis=0).flatten()


def test_mic_levels(seconds: float = 4.0) -> None:
    """Run this once and talk normally â€” tells you if SILENCE_THRESHOLD needs adjusting."""
    print(f"Recording {seconds:.0f}s test â€” talk normally now...")
    audio = record_from_mic(max_seconds=seconds, silence_stop=False)
    avg, peak = float(np.abs(audio).mean()), float(np.abs(audio).max())
    print(f"Average volume: {avg:.4f} | Peak volume: {peak:.4f}")
    if avg < SILENCE_THRESHOLD * 1.5:
        print(f"âš ï¸ Your speech volume is close to SILENCE_THRESHOLD ({SILENCE_THRESHOLD}) â€” "
              f"consider lowering it to about {avg/2:.4f}.")
    else:
        print("âœ… Volume looks fine relative to the current threshold.")


def save_wav(audio: np.ndarray, path: str, sample_rate: int = SAMPLE_RATE) -> None:
    pcm16 = np.clip(audio, -1.0, 1.0)
    pcm16 = (pcm16 * 32767).astype(np.int16)
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm16.tobytes())


class RealSTT:
    """Real microphone -> local Whisper transcription. Explicitly told the
    spoken language is Urdu so it transcribes accurately (native script),
    which is then transliterated to Roman Urdu by Groq in the reply step."""

    def transcribe(self, audio: np.ndarray) -> tuple[str, float]:
        start = time.perf_counter()
        result = _whisper_model.transcribe(audio, language="ur", task="transcribe", fp16=False)
        elapsed = time.perf_counter() - start
        return result["text"].strip(), elapsed


In [6]:
import json as _json
from groq import Groq

_groq_client = Groq(api_key=GROQ_API_KEY)

# Tried in order; free-tier model availability shifts, so we probe your actual key
# against a few candidates instead of hardcoding one name.
GROQ_MODEL_CANDIDATES = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
]
GROQ_MODEL = None  # resolved below


def _pick_working_groq_model() -> str:
    global GROQ_MODEL
    for candidate in GROQ_MODEL_CANDIDATES:
        try:
            _groq_client.chat.completions.create(
                model=candidate,
                messages=[{"role": "user", "content": "ping"}],
                max_tokens=5,
            )
            print(f"[Using Groq model: {candidate}]")
            GROQ_MODEL = candidate
            return candidate
        except Exception as e:
            print(f"[{candidate} unavailable ({e}) - trying next]")
    raise RuntimeError(
        "None of the candidate Groq models worked with this API key. "
        "Check console.groq.com for currently available free-tier model names and add one to "
        "GROQ_MODEL_CANDIDATES above."
    )


_pick_working_groq_model()

GROQ_SYSTEM_INSTRUCTION = """You are a warm, professional Pakistani real-estate sales executive
speaking natural Pakistani UrduLish (mixed Urdu/English). Rules:
- Keep replies short: 1-2 sentences, like a real phone call, not an essay.
- Acknowledge what the caller just said before asking a follow-up (e.g. "Ji bilkul...", "Acha...",
  "Ji samajh gaya...").
- Remember earlier details in this conversation (budget, area, bedrooms, prior options discussed)
  and use them naturally without asking the caller to repeat themselves.
- Handle objections (price, trust, location, investment, builder reputation, maintenance) with:
  Acknowledge -> Clarify -> Evidence -> Choice. Never invent discounts, guaranteed investment
  returns, or unverified builder/maintenance claims.
- If asked for legal documents, ownership verification, or anything requiring a human specialist,
  say you'll arrange a human/legal specialist rather than inventing details.
Always respond with a JSON object: {"roman_urdu": "<Roman Urdu transliteration of what the caller
just said>", "reply": "<your reply as the sales executive>"}."""


class RealConversationSession:
    """One real phone-call session.

    - Real memory (Task 3): full chat history sent to Groq each turn, PLUS your own
      ConversationMemory/LocationResolver (budget/city/area/bedrooms extraction) updated
      every turn regardless of whether Groq succeeds.
    - Real objection handling (Task 4): ObjectionHandler.classify() detects the objection
      type; on a Groq failure, ObjectionHandler.respond() builds a real structured reply
      instead of a generic line.
    - Retries once on a transient error, then falls back to a memory-and-objection-aware
      reply built from the above instead of crashing or repeating a canned line.
    """

    def __init__(self) -> None:
        self._messages = [{"role": "system", "content": GROQ_SYSTEM_INSTRUCTION}]
        # Reuses Task 3's classes (already defined above in this notebook)
        self._memory = ConversationMemory(location_resolver=LocationResolver(location_knowledge_base))
        # Reuses Task 4's classifier/response library (already defined above in this notebook)
        self._objection_handler = ObjectionHandler()
        self.last_roman_transcript = ""
        self.last_objection_type = None
        self.last_retry_wait_seconds = 0.0

    def reply(self, urdu_script_transcript: str) -> str:
        self.last_retry_wait_seconds = 0.0
        self._messages.append({"role": "user", "content": urdu_script_transcript})
        for attempt in range(2):
            try:
                completion = _groq_client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=self._messages,
                    response_format={"type": "json_object"},
                    max_tokens=300,
                    temperature=0.4,
                )
                text = (completion.choices[0].message.content or "").strip()
                if not text:
                    raise RuntimeError("empty response from Groq")
                data = _json.loads(text)
                roman = (data.get("roman_urdu") or "").strip()
                reply_text = (data.get("reply") or "").strip()
                if not reply_text:
                    raise RuntimeError("empty 'reply' field from Groq")

                self.last_roman_transcript = roman or urdu_script_transcript
                self._update_memory_and_objection(self.last_roman_transcript)
                self._messages.append({"role": "assistant", "content": text})
                return reply_text
            except Exception as e:
                if attempt == 0 and ("429" in str(e) or "503" in str(e) or "timeout" in str(e).lower()):
                    print("[Groq busy/rate-limited, retrying once...]")
                    wait_s = 1.5
                    self.last_retry_wait_seconds += wait_s
                    time.sleep(wait_s)
                    continue
                print(f"[Groq call failed ({e}) - falling back to memory/objection-aware response]")
                # Best-effort: try to extract memory/objection signal even from the raw
                # Urdu-script transcript (imperfect, but better than nothing); prior turns'
                # memory (from earlier successful calls) is preserved regardless.
                self._update_memory_and_objection(urdu_script_transcript)
                self.last_roman_transcript = urdu_script_transcript
                return self._fallback_reply(urdu_script_transcript)

    def _update_memory_and_objection(self, roman_ish_text: str) -> None:
        try:
            self._memory.update_from_user(roman_ish_text)
        except Exception:
            pass  # ambiguous location reference etc. â€” don't let memory bookkeeping crash a turn
        objection_type = self._objection_handler.classify(roman_ish_text)
        if objection_type is not None:
            self.last_objection_type = objection_type

    def _fallback_reply(self, transcript: str) -> str:
        # Objection takes priority when detected â€” gives a real structured reply, not a guess.
        if self.last_objection_type is not None:
            response = self._objection_handler.responses[self.last_objection_type]
            return f"{response.acknowledgement} {response.clarification}"

        state = self._memory.resolve(transcript)
        parts = []
        if state.get("area"):
            parts.append(f"{state['area']} mein")
        if state.get("budget_pkr"):
            crore = state["budget_pkr"] / 10_000_000
            parts.append(f"{crore:.1f} crore ke budget mein")

        if state.get("cheaper_than_previous") and parts:
            return f"Ji bilkul, {' '.join(parts)} hum aapko is se sasti options dikhate hain."
        if parts:
            return f"Ji, {' '.join(parts)} maine note kar liya hai â€” kya aap koi specific area ya bedroom count bhi bata sakte hain?"
        return "Ji, samajh gaya. Aapka preferred area aur approximate budget kya hai?"


[llama-3.3-70b-versatile unavailable (Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}) - trying next]
[Using Groq model: openai/gpt-oss-120b]


In [7]:
import io
import soundfile as sf


class RealFishAudioTTS:
    """Real Fish Audio API call -> plays audio through speakers and returns
    the raw audio bytes so the caller can save them for Task 5 recordings."""

    def synthesize(self, text: str) -> tuple[bytes, float]:
        if not FISH_AUDIO_API_KEY:
            raise RuntimeError("Fish Audio API key was empty â€” re-run the setup cell and paste your key.")
        start = time.perf_counter()
        response = requests.post(
            FISH_TTS_URL,
            headers={
                "Authorization": f"Bearer {FISH_AUDIO_API_KEY}",
                "Content-Type": "application/json",
                "model": FISH_MODEL,
            },
            json={"text": text, "format": "wav"},
            timeout=30,
        )
        response.raise_for_status()
        first_audio_latency = time.perf_counter() - start
        return response.content, first_audio_latency

    def play(self, audio_bytes: bytes) -> None:
        data, sr = sf.read(io.BytesIO(audio_bytes), dtype="float32")
        sd.play(data, sr)
        sd.wait()


In [8]:
def run_real_conversation_turn(session: "RealConversationSession", turn_index: int, session_id: str = "session01") -> dict:
    stt = RealSTT()
    tts = RealFishAudioTTS()

    transcript = ""
    audio_in = None
    for attempt in range(3):
        audio_in = record_from_mic()
        transcript, stt_latency = stt.transcribe(audio_in)
        if transcript.strip():
            break
        print("[No speech detected â€” let's try that turn again]")

    if not transcript.strip():
        raise RuntimeError("No speech detected after 3 attempts â€” check your microphone input.")

    # Latency clock starts here: from the moment you finish speaking, not from when the
    # recording began (which would count your own talking time as "system latency").
    t0 = time.perf_counter()

    caller_path = f"{RECORDINGS_DIR}/{session_id}_turn{turn_index:02d}_caller.wav"
    save_wav(audio_in, caller_path)

    response_text = session.reply(transcript)
    print(f"You said (Roman Urdu): {session.last_roman_transcript}")
    print(f"Agent: {response_text}")

    audio_bytes, _ = tts.synthesize(response_text)
    agent_path = f"{RECORDINGS_DIR}/{session_id}_turn{turn_index:02d}_agent.wav"
    with open(agent_path, "wb") as f:
        f.write(audio_bytes)

    # Stop the clock here â€” audio is ready to play. Playback duration and the person's own
    # speaking time are not part of the system's response latency.
    raw_elapsed = time.perf_counter() - t0
    elapsed = max(0.0, raw_elapsed - session.last_retry_wait_seconds)

    tts.play(audio_bytes)  # actually play it now, after the timer has stopped

    return {
        "transcript_roman_urdu": session.last_roman_transcript,
        "transcript_urdu_script": transcript,
        "response_text": response_text,
        "caller_recording": caller_path,
        "agent_recording": agent_path,
        "first_audio_latency_ms": round(elapsed * 1000, 1),
        "end_to_end_latency_ms": round(elapsed * 1000, 1),
        "retry_wait_excluded_ms": round(session.last_retry_wait_seconds * 1000, 1),
    }


def _load_mono_at_rate(path: str, target_rate: int = SAMPLE_RATE) -> np.ndarray:
    """Reads any wav file and resamples (simple linear interpolation, no extra
    dependency needed) to target_rate mono float32, so caller clips (16kHz) and
    agent/TTS clips (often a different rate) can be concatenated cleanly."""
    data, sr = sf.read(path, dtype="float32")
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != target_rate:
        duration = len(data) / sr
        target_len = int(duration * target_rate)
        old_x = np.linspace(0, duration, num=len(data), endpoint=False)
        new_x = np.linspace(0, duration, num=target_len, endpoint=False)
        data = np.interp(new_x, old_x, data).astype("float32")
    return data


def stitch_conversation_recording(results: list[dict], session_id: str, gap_seconds: float = 0.4) -> str:
    """Combines every turn's caller + agent clips, in order, into one continuous
    .wav file so a reviewer can listen through the whole conversation at once
    instead of opening separate per-turn files."""
    gap = np.zeros(int(SAMPLE_RATE * gap_seconds), dtype="float32")
    pieces = []
    for turn in results:
        pieces.append(_load_mono_at_rate(turn["caller_recording"]))
        pieces.append(gap)
        pieces.append(_load_mono_at_rate(turn["agent_recording"]))
        pieces.append(gap)
    full_audio = np.concatenate(pieces) if pieces else np.zeros(0, dtype="float32")
    combined_path = f"{RECORDINGS_DIR}/{session_id}_full_conversation.wav"
    save_wav(full_audio, combined_path)
    return combined_path


def run_real_conversation(num_turns: int = 5, session_id: str = "session01") -> list[dict]:
    """Run one full live conversation (one Groq session = one Task 5 recording) for
    real-voice Task 5 evaluation. Prompts before each turn so you control pacing.
    Context carries turn-to-turn via both the Groq chat history and your own
    ConversationMemory/ObjectionHandler (Task 3/4), which also back the fallback path.
    At the end, automatically stitches all turns into one combined .wav for reviewers."""
    session = RealConversationSession()
    results = []
    for i in range(1, num_turns + 1):
        input(f"\nPress Enter, then speak turn {i}/{num_turns}...")
        results.append(run_real_conversation_turn(session, i, session_id))

    combined_path = stitch_conversation_recording(results, session_id)
    print(f"\nFull conversation saved to: {combined_path}")
    for r in results:
        r["combined_recording"] = combined_path
    return results


# Example â€” run one of your 10 Task 5 scenarios (uncomment to run):
results = run_real_conversation(num_turns=3, session_id="conversation_01_normal")
for r in results:
    print(r)


[recording... speak now, up to 10s]
[recording stopped]
You said (Roman Urdu): Islam Alaikum, mujhe Hoor mein 3 crore ke budget ka koi achi property chahiye
Agent: Ji bilkul, Hoor area mein 3 crore ke budget ke andar 2-3 BHK options available hain; aapko koi specific size ya amenities chahiye?
[recording... speak now, up to 10s]
[recording stopped]
[Groq call failed (Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}) - falling back to memory/objection-aware response]
You said (Roman Urdu): ÛŒÛ ÛÙˆ Ú©Û ÙØ±Ù†Ù„ÛŒ ÙØ±Ù†Ù„ÛŒ Ø§Ø±ÛŒÛ ÛÙˆ Ø§ÙˆØ± Ù¾Ø§Ú©Ø³ Ùˆ Ú¯ÛŒØ±Û Ù¾Ø§Ø³ ÛÙˆÚºØŒ ÛÙˆØ³Ù¾Ù¹Ù„Ø² Ùˆ Ú¯ÛŒØ±Û Ù¾Ø§Ø³ ÛÙˆÚº
Agent: Ji, budget concern bilkul samajh sakta hoon. Aapka comfortable budget ya sab se important priority kya hai?
[recording... speak now, up to 10s]
[recording stopped]
[Groq call failed (E

### Running this for real

- This must run in **Jupyter/VS Code on your own laptop**, not a hosted/sandboxed notebook environment â€” it needs real access to your microphone and speakers.
- Run the setup cell first â€” it prompts (hidden input) for your Groq and Fish Audio keys. Nothing gets written into the notebook file, so it's safe to submit/upload as-is.
- Run `test_mic_levels()` once if you're unsure about `SILENCE_THRESHOLD` â€” it tells you directly whether to lower it.
- For each of your 10 Task 5 scenarios, call `run_real_conversation(num_turns=..., session_id="conversation_01_normal")` (etc.). Each call opens one fresh session with real memory (Groq chat history + `ConversationMemory`), saves per-turn caller/agent `.wav` files, and automatically stitches them into one `..._full_conversation.wav` at the end â€” that's the single file to hand each reviewer per scenario.
- **Latency numbers now reflect real system turnaround**, not your speaking time, TTS playback duration, or deliberate retry-backoff waits â€” check `retry_wait_excluded_ms` in each turn's result if you want to see how much backoff time was excluded.
- **If Groq is rate-limited or briefly errors:** the pipeline retries once, then falls back to a reply built from your own `ConversationMemory` state and any detected `ObjectionHandler` objection type â€” so even a failed call still produces a context-aware reply instead of a generic one.
- If `sounddevice` can't find a microphone/speaker, check your OS sound settings.
- If Whisper feels slow, drop `WHISPER_MODEL_SIZE` back to `"base"` (less accurate on Urdu) or try `"tiny"`.
- If Fish Audio returns 401, your key is wrong; 402 means your free-tier access has changed â€” check your dashboard balance.
- If Groq returns a "model decommissioned"/"not found" error, add the new model name to the top of `GROQ_MODEL_CANDIDATES` in the Groq cell.


# Task 5 â€” Human Evaluation: Real Voice-Agent Quality

This section upgrades Task 5 from a synthetic fixture benchmark into a **proper human-evaluation protocol**.

The evaluation separates:

1. **Human-rated quality:** Naturalness, Persuasiveness, Fluency, Voice Quality, and Conversation Flow.
2. **Objective system measurements:** time-to-first-audio, end-to-end latency, interruption recovery latency, and successful interruption recovery.
3. **Evidence:** every scored conversation must have a real recording/transcript reference and two independent reviewers.

> **Important:** The notebook does not invent human scores. Scores remain `None` until reviewers listen to the actual recordings and enter their ratings. Synthetic WAV fixtures from earlier testing are explicitly excluded from the human-quality results.


In [9]:
from dataclasses import dataclass, asdict
from pathlib import Path
from statistics import mean, median
import json
import math
import csv

EVALUATION_DIMENSIONS = (
    "naturalness",
    "persuasiveness",
    "fluency",
    "voice_quality",
    "conversation_flow",
)

SCORING_GUIDE = {
    "naturalness": {
        1: "Robotic/scripted; unnatural turn-taking or inappropriate fillers.",
        3: "Generally natural but noticeable robotic moments.",
        5: "Feels like a natural Pakistani sales conversation with appropriate UrduLish."
    },
    "persuasiveness": {
        1: "Pushy, unclear, misleading, or fails to address the concern.",
        3: "Useful but generic or inconsistently persuasive.",
        5: "Helpful, relevant, evidence-based, and low-pressure."
    },
    "fluency": {
        1: "Frequent awkward pauses, pronunciation problems, or broken code-switching.",
        3: "Mostly fluent with some noticeable issues.",
        5: "Smooth UrduLish, pronunciation, pacing, fillers, and sentence delivery."
    },
    "voice_quality": {
        1: "Poor audio, artifacts, inconsistent volume, or difficult to understand.",
        3: "Understandable with minor artifacts.",
        5: "Clear, consistent, natural-sounding voice."
    },
    "conversation_flow": {
        1: "Loses context, repeats itself, or responds incorrectly.",
        3: "Mostly coherent but has occasional flow/context issues.",
        5: "Maintains context, handles interruptions/objections, and moves naturally to the next step."
    },
}

SCENARIOS = [
    ("REC-001", "normal property inquiry", "Tests greeting, qualification, naturalness, and flow."),
    ("REC-002", "budget objection", "Tests price concern handling and alternatives."),
    ("REC-003", "trust objection", "Tests trust/builder verification and escalation."),
    ("REC-004", "investment question", "Tests factual investment language and safety."),
    ("REC-005", "builder concern", "Tests verification and unsupported-claim avoidance."),
    ("REC-006", "maintenance question", "Tests clarification and evidence-based answers."),
    ("REC-007", "interruption and barge-in", "Tests interruption detection and recovery."),
    ("REC-008", "context-memory conversation", "Tests 3-turn memory: budget â†’ DHA â†’ cheaper option."),
    ("REC-009", "UrduLish-heavy inquiry", "Tests Pakistani UrduLish, code-switching, and pronunciation."),
    ("REC-010", "human handoff", "Tests respectful escalation to a human specialist."),
]

@dataclass
class HumanEvaluationRecord:
    conversation_id: str
    reviewer_id: str
    recording_path: str
    scenario: str
    naturalness: int | None = None
    persuasiveness: int | None = None
    fluency: int | None = None
    voice_quality: int | None = None
    conversation_flow: int | None = None
    first_audio_latency_ms: float | None = None
    end_to_end_latency_ms: float | None = None
    interruption_recovery_ms: float | None = None
    interruption_recovered: bool | None = None
    transcript_notes: str = ""
    strengths: str = ""
    improvements: str = ""

    def validate(self, allow_pending: bool = True):
        for dimension in EVALUATION_DIMENSIONS:
            value = getattr(self, dimension)
            if value is None and allow_pending:
                continue
            if value is None or not 1 <= value <= 5:
                raise ValueError(f"{dimension} must be an integer from 1 to 5.")
        for field_name in (
            "first_audio_latency_ms",
            "end_to_end_latency_ms",
            "interruption_recovery_ms",
        ):
            value = getattr(self, field_name)
            if value is not None and value < 0:
                raise ValueError(f"{field_name} cannot be negative.")
        if not self.conversation_id or not self.reviewer_id or not self.scenario:
            raise ValueError("conversation_id, reviewer_id, and scenario are required.")

    def complete(self):
        return all(getattr(self, dimension) is not None for dimension in EVALUATION_DIMENSIONS)

    def quality_score(self):
        if not self.complete():
            return None
        return round(mean(getattr(self, dimension) for dimension in EVALUATION_DIMENSIONS), 2)

# Two independent reviewers are assigned to every real recording.
# Scores are intentionally blank until the reviewers actually listen.
evaluation_queue = [
    HumanEvaluationRecord(
        conversation_id=conversation_id,
        reviewer_id=reviewer_id,
        recording_path=f"REAL_RECORDINGS/{conversation_id}.wav",
        scenario=scenario,
    )
    for conversation_id, scenario, _ in SCENARIOS
    for reviewer_id in ("reviewer-urdu-01", "reviewer-urdu-02")
]

for record in evaluation_queue:
    record.validate(allow_pending=True)

print(f"Evaluation queue created: {len(evaluation_queue)} reviewer-record pairs")
print(f"Conversations: {len(SCENARIOS)}")
print("Each conversation has two independent reviewers.")


Evaluation queue created: 20 reviewer-record pairs
Conversations: 10
Each conversation has two independent reviewers.


## Reviewer Rubric

Each reviewer scores **1â€“5 independently before discussing the call**.

| Dimension | 1 | 3 | 5 |
|---|---|---|---|
| Naturalness | Robotic/scripted | Mostly natural | Realistic Pakistani sales conversation |
| Persuasiveness | Pushy/misleading | Useful but generic | Helpful, relevant, low-pressure |
| Fluency | Awkward/broken | Mostly smooth | Smooth UrduLish + natural pacing |
| Voice Quality | Poor/artifacts | Acceptable | Clear and consistent |
| Conversation Flow | Loses context | Mostly coherent | Strong context + turn-taking |

### What reviewers should specifically listen for

- `Ji bilkul`, `Acha`, `Hmm...`, and similar cues should sound **contextual**, not randomly inserted.
- Pauses should feel like thinking rather than audio gaps.
- Laughter should only occur in appropriate positive/light moments.
- The agent should stop or yield promptly when interrupted.
- The agent should remember earlier budget/location requirements.
- Objections should be acknowledged rather than argued with.
- Investment/builder/trust answers must avoid unsupported claims or guarantees.
- UrduLish code-switching should sound natural rather than translated word-for-word.


In [10]:
def save_evaluation_template(records, path="task5_human_evaluation_template.csv"):
    fieldnames = [
        "conversation_id", "reviewer_id", "recording_path", "scenario",
        *EVALUATION_DIMENSIONS,
        "first_audio_latency_ms", "end_to_end_latency_ms",
        "interruption_recovery_ms", "interruption_recovered",
        "transcript_notes", "strengths", "improvements",
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(fieldnames=fieldnames, f=f)
        writer.writeheader()
        for record in records:
            writer.writerow(asdict(record))
    return Path(path)

template_path = save_evaluation_template(evaluation_queue)
print(f"Reviewer template: {template_path.resolve()}")


Reviewer template: F:\Web 3 Geeks Internship\Week 4\Day 3\task5_human_evaluation_template.csv


In [11]:
def load_completed_evaluations(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Evaluation file not found: {path}")

    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            def score(name):
                value = row.get(name, "").strip()
                return int(value) if value else None

            def number(name):
                value = row.get(name, "").strip()
                return float(value) if value else None

            def boolean(name):
                value = row.get(name, "").strip().casefold()
                if value in ("true", "1", "yes"):
                    return True
                if value in ("false", "0", "no"):
                    return False
                return None

            record = HumanEvaluationRecord(
                conversation_id=row["conversation_id"],
                reviewer_id=row["reviewer_id"],
                recording_path=row["recording_path"],
                scenario=row["scenario"],
                naturalness=score("naturalness"),
                persuasiveness=score("persuasiveness"),
                fluency=score("fluency"),
                voice_quality=score("voice_quality"),
                conversation_flow=score("conversation_flow"),
                first_audio_latency_ms=number("first_audio_latency_ms"),
                end_to_end_latency_ms=number("end_to_end_latency_ms"),
                interruption_recovery_ms=number("interruption_recovery_ms"),
                interruption_recovered=boolean("interruption_recovered"),
                transcript_notes=row.get("transcript_notes", ""),
                strengths=row.get("strengths", ""),
                improvements=row.get("improvements", ""),
            )
            record.validate(allow_pending=True)
            rows.append(record)
    return rows

print("The loader is ready for the CSV after the real recordings have been reviewed.")


The loader is ready for the CSV after the real recordings have been reviewed.


In [12]:
def summarize_evaluation(records):
    if not records:
        raise ValueError("No evaluation records supplied.")

    completed = [r for r in records if r.complete()]
    summary = {
        "total_review_records": len(records),
        "completed_review_records": len(completed),
        "reviewers": len({r.reviewer_id for r in records}),
        "conversations": len({r.conversation_id for r in records}),
        "average_scores": {},
        "average_quality_score": None,
        "median_first_audio_latency_ms": None,
        "p95_first_audio_latency_ms": None,
        "median_end_to_end_latency_ms": None,
        "p95_end_to_end_latency_ms": None,
        "interruption_recovery_rate": None,
        "records_below_4": [],
    }

    if not completed:
        return summary

    for dimension in EVALUATION_DIMENSIONS:
        summary["average_scores"][dimension] = round(
            mean(getattr(r, dimension) for r in completed), 2
        )

    summary["average_quality_score"] = round(
        mean(r.quality_score() for r in completed), 2
    )

    first_audio = [
        r.first_audio_latency_ms for r in completed
        if r.first_audio_latency_ms is not None
    ]
    end_to_end = [
        r.end_to_end_latency_ms for r in completed
        if r.end_to_end_latency_ms is not None
    ]
    interruption = [
        r.interruption_recovered for r in completed
        if r.interruption_recovered is not None
    ]

    if first_audio:
        summary["median_first_audio_latency_ms"] = round(median(first_audio), 1)
        summary["p95_first_audio_latency_ms"] = round(
            sorted(first_audio)[max(0, math.ceil(0.95 * len(first_audio)) - 1)], 1
        )
    if end_to_end:
        summary["median_end_to_end_latency_ms"] = round(median(end_to_end), 1)
        summary["p95_end_to_end_latency_ms"] = round(
            sorted(end_to_end)[max(0, math.ceil(0.95 * len(end_to_end)) - 1)], 1
        )
    if interruption:
        summary["interruption_recovery_rate"] = round(mean(interruption), 3)

    for r in completed:
        weak = [d for d in EVALUATION_DIMENSIONS if getattr(r, d) < 4]
        if weak:
            summary["records_below_4"].append({
                "conversation_id": r.conversation_id,
                "reviewer_id": r.reviewer_id,
                "dimensions": weak,
            })

    return summary

# Pending by design: no invented human scores.
pending_summary = summarize_evaluation(evaluation_queue)
print(json.dumps(pending_summary, indent=2))
assert pending_summary["completed_review_records"] == 0
print("\nPASS: Task 5 refuses to manufacture human-evaluation results.")


{
  "total_review_records": 20,
  "completed_review_records": 0,
  "reviewers": 2,
  "conversations": 10,
  "average_scores": {},
  "average_quality_score": null,
  "median_first_audio_latency_ms": null,
  "p95_first_audio_latency_ms": null,
  "median_end_to_end_latency_ms": null,
  "p95_end_to_end_latency_ms": null,
  "interruption_recovery_rate": null,
  "records_below_4": []
}

PASS: Task 5 refuses to manufacture human-evaluation results.


## Reviewer Agreement

Two independent reviewers are used for every recording. This makes the result more credible than a single self-score.

The notebook reports disagreement rather than hiding it. A large disagreement should trigger a short calibration discussion and a second review of the exact turn.

The following function calculates **exact agreement** and **mean absolute disagreement** for each dimension after both reviewers have submitted scores.


In [13]:
def reviewer_agreement(records):
    by_key = {}
    for r in records:
        if not r.complete():
            continue
        by_key.setdefault(r.conversation_id, {})[r.reviewer_id] = r

    results = {}
    for dimension in EVALUATION_DIMENSIONS:
        pairs = []
        for conversation_id, reviewers in by_key.items():
            values = list(reviewers.values())
            if len(values) < 2:
                continue
            a, b = values[0], values[1]
            sa, sb = getattr(a, dimension), getattr(b, dimension)
            pairs.append((sa, sb))

        if not pairs:
            results[dimension] = {
                "paired_conversations": 0,
                "exact_agreement_rate": None,
                "mean_absolute_difference": None,
            }
            continue

        results[dimension] = {
            "paired_conversations": len(pairs),
            "exact_agreement_rate": round(
                mean(a == b for a, b in pairs), 3
            ),
            "mean_absolute_difference": round(
                mean(abs(a - b) for a, b in pairs), 3
            ),
        }
    return results

print("Agreement report will populate automatically after the CSV is completed.")


Agreement report will populate automatically after the CSV is completed.


## Objective Latency Gate

Human reviewers can rate whether latency *feels* natural, but latency itself should be measured from timestamps.

Recommended release evidence:

- **p50 time-to-first-audio:** target `< 1.0 s`
- **p95 time-to-first-audio:** target `< 2.0 s`
- **p50 end-to-end turn latency:** target `< 2.0 s`
- **Interruption recovery:** agent should stop/yield promptly after caller speech is detected
- Report provider/model/voice/prompt/network conditions with the measurements.

These thresholds are evaluation targets for this task, not claims about the current production system.


In [14]:
def latency_gate(summary):
    p50 = summary["median_first_audio_latency_ms"]
    p95 = summary["p95_first_audio_latency_ms"]
    recovery = summary["interruption_recovery_rate"]

    if p50 is None or p95 is None:
        return {
            "status": "PENDING_REAL_RECORDINGS",
            "reason": "Objective latency measurements are not available yet."
        }

    checks = {
        "p50_first_audio_under_1000ms": p50 < 1000,
        "p95_first_audio_under_2000ms": p95 < 2000,
        "interruption_recovery_100_percent": (
            recovery is not None and recovery == 1.0
        ),
    }
    return {
        "status": "PASS" if all(checks.values()) else "REVIEW",
        "checks": checks,
    }

print(latency_gate(pending_summary))


{'status': 'PENDING_REAL_RECORDINGS', 'reason': 'Objective latency measurements are not available yet.'}


## Final Human-Evaluation Procedure

### Step 1 â€” Record real conversations

Use the actual **STT â†’ LLM â†’ TTS** voice pipeline. Record all ten scenarios above. Do not use the synthetic PCM fixtures as evidence of voice quality.

### Step 2 â€” Blind review

Give the same recordings to `reviewer-urdu-01` and `reviewer-urdu-02`. Reviewers should not see each other's scores.

### Step 3 â€” Score independently

Each reviewer enters:

- Naturalness: 1â€“5
- Persuasiveness: 1â€“5
- Fluency: 1â€“5
- Voice Quality: 1â€“5
- Conversation Flow: 1â€“5
- First-audio latency: measured milliseconds
- End-to-end latency: measured milliseconds
- Interruption recovery: yes/no
- Strengths and improvements

### Step 4 â€” Aggregate

Run the summary and agreement cells. The notebook will calculate:

- Average score per dimension
- Overall average human-quality score
- p50/p95 latency
- Interruption recovery rate
- Reviewer agreement
- Conversations/dimensions requiring follow-up

### Step 5 â€” Quality gate

A strong final submission should demonstrate:

- No dimension below 4/5 without documented remediation
- p95 first-audio latency under 2 seconds
- Successful interruption recovery
- Context memory demonstrated in the recorded call
- Objection handling demonstrated in the recorded calls
- Natural Pakistani UrduLish delivery
- No fabricated claims or guaranteed investment outcomes

**This structure makes Task 5 evidence-based instead of relying on self-reported or synthetic scores.**
